In [39]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [40]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [41]:
df.shape

(569, 33)

In [42]:
df.drop(['id', 'Unnamed: 32'], inplace=True, axis=1)
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [43]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('diagnosis', axis=1), df['diagnosis'], test_size=0.2, random_state=42)

In [44]:
scaler = StandardScaler();
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [45]:
lebelEncoder = LabelEncoder()
y_train = lebelEncoder.fit_transform(y_train)
y_test = lebelEncoder.transform(y_test)

In [46]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [47]:
class CustomeDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, index):
    return self.features[index], self.labels[index]


In [48]:
train_dataset = CustomeDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomeDataset(X_test_tensor, y_test_tensor)

In [49]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [50]:
class Model(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(num_features, 32),
        nn.ReLU(),
        nn.Linear(32, 1),
        nn.Sigmoid()
    )

  def forward(self, X):
    return self.network(X)

In [51]:
loss_function = nn.BCELoss()
epochs = 100

In [52]:
model = Model(X_train_tensor.shape[1])
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
  for X_batch, y_batch in train_loader:
    y_pred = model.forward(X_batch)

    loss = loss_function(y_pred, y_batch.view(-1, 1))

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    print(f'Epoch: {epoch+1}, Loss: {loss.item():.4f}')

Epoch: 1, Loss: 0.6649
Epoch: 1, Loss: 0.6326
Epoch: 1, Loss: 0.6362
Epoch: 1, Loss: 0.6437
Epoch: 1, Loss: 0.6342
Epoch: 1, Loss: 0.6126
Epoch: 1, Loss: 0.5985
Epoch: 1, Loss: 0.5846
Epoch: 2, Loss: 0.5733
Epoch: 2, Loss: 0.5837
Epoch: 2, Loss: 0.5708
Epoch: 2, Loss: 0.5816
Epoch: 2, Loss: 0.5342
Epoch: 2, Loss: 0.5254
Epoch: 2, Loss: 0.5319
Epoch: 2, Loss: 0.5684
Epoch: 3, Loss: 0.5072
Epoch: 3, Loss: 0.5367
Epoch: 3, Loss: 0.4830
Epoch: 3, Loss: 0.5113
Epoch: 3, Loss: 0.4754
Epoch: 3, Loss: 0.4997
Epoch: 3, Loss: 0.4695
Epoch: 3, Loss: 0.4404
Epoch: 4, Loss: 0.4190
Epoch: 4, Loss: 0.4649
Epoch: 4, Loss: 0.4667
Epoch: 4, Loss: 0.4596
Epoch: 4, Loss: 0.4547
Epoch: 4, Loss: 0.3988
Epoch: 4, Loss: 0.4460
Epoch: 4, Loss: 0.3821
Epoch: 5, Loss: 0.4118
Epoch: 5, Loss: 0.3829
Epoch: 5, Loss: 0.4136
Epoch: 5, Loss: 0.4084
Epoch: 5, Loss: 0.3787
Epoch: 5, Loss: 0.3739
Epoch: 5, Loss: 0.4094
Epoch: 5, Loss: 0.3168
Epoch: 6, Loss: 0.4310
Epoch: 6, Loss: 0.3740
Epoch: 6, Loss: 0.3095
Epoch: 6, L

In [53]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        y_pred = model(X_batch)

        y_pred = (y_pred > 0.5).float()

        correct += (
            y_pred.squeeze()
            ==
            y_batch
        ).sum().item()

        total += y_batch.size(0)

print(f'Accuracy: {correct/total:.4f}')

Accuracy: 0.9825
